# 06 · Agents and memory

After this notebook you can build a tool-calling agent with LangChain 1.x's `create_agent`, read and stream its
message trace, give it memory across turns with a checkpointer and thread ids, add middleware (logging, call limits,
PII redaction, human approval, summarisation), keep chat history for plain chains, and say when to drop down to
LangGraph.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min · **Needs:** [03 · Structured output and tools](03_structured_output_and_tools.ipynb) (the manual tool loop)

## Why this matters in interviews

- "What is an agent and how is it different from a chain?" — a chain's steps are fixed by you; an agent's model
  decides the next step (which tool, or stop) in a loop. You should be able to show the loop in the trace.
- Memory questions ("the agent forgot everything after 50 turns") are about *what is resent to the model*: threads,
  checkpoints, trimming and summarisation — and the trap that summaries are lossy.
- Production questions — step budgets, cost guards, PII, human approval before irreversible actions — are exactly
  what LangChain 1.x middleware packages up.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag01` | What is an AI agent, and how is it different from a chain? |
| `ag02` | What is the core agent loop? |
| `ag20` | What is agent memory, and what are its layers? |
| `ag15` | An agent has been running for 50 turns and suddenly forgets everything from the beginning. What happened? |
| `pr11` | What is conversation summarisation, and what is the trap? |
| `ag16` | How do you stop an agent looping forever? |
| `tl22` | Give me the four guards that stop an agent costing $400 in one run. |
| `ag14` | What is human in the loop, and where does it belong? |
| `ag25` | How do you handle irreversible actions? |
| `ag40` | What is the difference between an agent framework and rolling your own? |

## Setup

Real code: `ChatOpenAI(model="gpt-4.1-mini", temperature=0)`; `langchain_chat_model()` returns it for the active
provider. **How the offline stand-in behaves as an agent brain:** it picks the tool whose name and description share
the most words with the user message (several tools if several match), fills the arguments from the message, and once
tool results are present answers "Based on the tool results: ...". A real model phrases the final answer naturally.
The demos are designed around that, so every trace below is what a real model would plausibly do on these inputs.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER
from llm_setup import langchain_chat_model, OFFLINE

import warnings
import matplotlib.pyplot as plt
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

llm = langchain_chat_model(temperature=0)

def show_trace(messages, width=95):
    """One line per message: type, text, tool calls."""
    for m in messages:
        calls = [f"{tc['name']}({tc['args']})" for tc in getattr(m, "tool_calls", [])]
        text = (m.content or "").replace("\n", " ")[:width]
        print(f"  {type(m).__name__:13s} {text} {' '.join(calls)}")

## 1. `create_agent`: the tool loop, prebuilt

`create_agent(model, tools=[...], system_prompt=...)` returns a compiled LangGraph graph running exactly the loop
from notebook 03: model → tool calls? → run tools → model again → ... → final answer. Input and output are a dict of
state with a `messages` list.

In LangChain 0.x this was `AgentExecutor` + `create_react_agent`/`initialize_agent`; those now live in
`langchain_classic.agents`. New code should use `create_agent`.

In [ ]:
ORDERS = {"A5521": "shipped on 2026-09-20, arriving 2026-09-24", "B7310": "processing, not shipped yet"}

@tool
def get_order_status(order_id: str) -> str:
    """Look up the shipping status of a customer order by its order id."""
    return ORDERS.get(order_id.upper(), f"no order with id {order_id}")

@tool
def search_policy(query: str) -> str:
    """Search the store policy documents (refund, returns, warranty) and return the matching policy text."""
    return "Refund policy: unused items can be returned within 30 days of delivery for a full refund."

agent = create_agent(llm, tools=[get_order_status, search_policy],
                     system_prompt="You are a support agent for an online store. Use the tools; never guess order data.")
print("agent type:", type(agent).__name__)

result = agent.invoke({"messages": [{"role": "user", "content": "Where is my order A5521?"}]})
show_trace(result["messages"])
assert [type(m).__name__ for m in result["messages"]] == ["HumanMessage", "AIMessage", "ToolMessage", "AIMessage"]

Read the trace like an interviewer would: the model's first `AIMessage` has no text, only a tool call; the
`ToolMessage` is **your** code's output; the last `AIMessage` is the answer. Two model calls for one question — each
one costs tokens and latency.

A question that needs two tools gets two calls in one step (parallel tool calls), and a question that needs none is
answered directly:

In [ ]:
both = agent.invoke({"messages": [{"role": "user", "content": "Where is my order B7310 and what is your refund policy?"}]})
show_trace(both["messages"])
calls = both["messages"][1].tool_calls
assert {tc["name"] for tc in calls} == {"get_order_status", "search_policy"}

print()
show_trace(agent.invoke({"messages": [{"role": "user", "content": "Hello there"}]})["messages"])

The graph itself: a `model` node, a `tools` node, and a conditional edge. The ASCII drawing does not show the edge
back from `tools` to `model`, so we also print the edge list — that back edge is the loop.

In [ ]:
graph = agent.get_graph()
print("nodes:", list(graph.nodes))
for e in graph.edges:
    print(f"  {e.source:>9s} -> {e.target:9s}{'   (conditional)' if e.conditional else ''}")
assert any(e.source == "tools" and e.target == "model" for e in graph.edges)

## 2. Streaming the agent's steps

| `stream_mode=` | Yields | Use it for |
|---|---|---|
| `"updates"` | `{node_name: state_update}` after each step | progress UI ("looking up your order..."), logging each step |
| `"messages"` | `(message_chunk, metadata)` token by token, from every model call | typing effect in a chat UI |
| `"values"` | the full state after each step | debugging |

In [ ]:
question = {"messages": [{"role": "user", "content": "Where is my order B7310?"}]}
for step in agent.stream(question, stream_mode="updates"):
    for node, update in step.items():
        for m in update["messages"]:
            what = [tc["name"] for tc in m.tool_calls] if getattr(m, "tool_calls", None) else (m.content or "")[:60]
            print(f"step from {node:6s}: {type(m).__name__:12s} {what}")

tokens = [chunk.content for chunk, meta in agent.stream(question, stream_mode="messages")
          if meta["langgraph_node"] == "model" and isinstance(chunk.content, str) and chunk.content]
print("\ntokens streamed from the model node:", tokens[:8], "...")
assert len(tokens) > 3

## 3. Memory across turns: a checkpointer and a `thread_id`

**In plain English:** the model is stateless (notebook 01). Memory means *something* stores the conversation and
resends it. With `create_agent`, pass a **checkpointer**: after every step it saves the whole state under the
`thread_id` you give in `config`. Next call with the same thread id → the history is loaded and your new message is
appended. A different thread id → a fresh conversation.

| Layer | What | In LangChain |
|---|---|---|
| short-term (thread) memory | this conversation's messages | checkpointer + `thread_id` (`InMemorySaver`; `SqliteSaver` / `PostgresSaver` in production) |
| long-term memory | facts across conversations (preferences, profile) | a LangGraph `store` (`create_agent(..., store=...)`), or your own DB/vector store as a tool |
| the context window | what the model actually sees this call | whatever the state holds, after trimming / summarising middleware |

In [ ]:
mem_agent = create_agent(llm, tools=[get_order_status], system_prompt="You are a helpful support agent.",
                         checkpointer=InMemorySaver())
alice = {"configurable": {"thread_id": "alice-1"}}
bob = {"configurable": {"thread_id": "bob-1"}}

mem_agent.invoke({"messages": [{"role": "user", "content": "My name is Alice."}]}, alice)
r_alice = mem_agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, alice)
r_bob = mem_agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, bob)
print("thread alice-1:", r_alice["messages"][-1].content, f"({len(r_alice['messages'])} messages in state)")
print("thread bob-1  :", r_bob["messages"][-1].content)
assert "Alice" in r_alice["messages"][-1].content and "Alice" not in r_bob["messages"][-1].content

state = mem_agent.get_state(alice)
print("saved state   :", len(state.values["messages"]), "messages | checkpoints so far:",
      len(list(mem_agent.get_state_history(alice))))

`InMemorySaver` lives in the Python process — restart and it is gone. Swap in a database checkpointer and the same
code survives restarts; the checkpoint history also lets you replay or fork a conversation
([checkpointing and time travel](../14_langgraph/03_checkpointing_memory_and_time_travel.ipynb)).

## 4. Middleware (LangChain 1.x)

**In plain English:** middleware are hooks around the agent loop — before/after each model call, around each tool
call — where you put the production concerns without touching the loop itself. This version ships (verified by
import below):

| Middleware | What it does |
|---|---|
| `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware` | step / cost budgets per run or per thread |
| `SummarizationMiddleware` | summarise old messages when the history gets long |
| `HumanInTheLoopMiddleware` | pause before chosen tools until a human approves, edits or rejects |
| `PIIMiddleware` | redact / mask / hash / block emails, cards, IPs... in inputs, outputs or tool results |
| `ModelRetryMiddleware`, `ModelFallbackMiddleware`, `ToolRetryMiddleware`, `ToolErrorMiddleware` | resilience |
| `LLMToolSelectorMiddleware`, `ContextEditingMiddleware`, `TodoListMiddleware`, ... | tool pre-selection, clearing old tool output, planning |
| `@before_model`, `@after_model`, `@wrap_model_call`, `@wrap_tool_call`, `@dynamic_prompt` | write your own |

In [ ]:
import langchain.agents.middleware as mw
available = sorted(n for n in dir(mw) if n.endswith("Middleware") and n != "AgentMiddleware")   # minus the base class
print(len(available), "built-in middleware classes:", ", ".join(available))
for needed in ["ModelCallLimitMiddleware", "SummarizationMiddleware", "HumanInTheLoopMiddleware", "PIIMiddleware"]:
    assert needed in available

### Your own middleware: see every hook fire

In [ ]:
from langchain.agents.middleware import before_model, after_model, wrap_tool_call

events = []

@before_model
def log_before(state, runtime):
    events.append(f"before_model   : model will see {len(state['messages'])} messages")

@after_model
def log_after(state, runtime):
    last = state["messages"][-1]
    events.append(f"after_model    : tool_calls={[tc['name'] for tc in last.tool_calls]}")

@wrap_tool_call
def log_tool(request, handler):
    events.append(f"wrap_tool_call : running {request.tool_call['name']}{request.tool_call['args']}")
    return handler(request)                       # call the tool (you could retry, time or block it here)

logged = create_agent(llm, tools=[get_order_status], middleware=[log_before, log_after, log_tool])
logged.invoke({"messages": [{"role": "user", "content": "Where is my order A5521?"}]})
print("\n".join(events))
assert len(events) == 5

### A cost guard: `ModelCallLimitMiddleware`

An agent that keeps calling tools can burn through a budget in one run (`tl22`, `ag16`). A run limit stops it. With
`exit_behavior="end"` the run ends gracefully with a message; `"error"` raises. Here the limit is 1 model call, so the
agent is stopped right after its first tool round, before it can write the answer:

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware

capped = create_agent(llm, tools=[get_order_status],
                      middleware=[ModelCallLimitMiddleware(run_limit=1, exit_behavior="end")])
capped_run = capped.invoke({"messages": [{"role": "user", "content": "Where is my order A5521?"}]})
show_trace(capped_run["messages"])
assert "limit" in capped_run["messages"][-1].content.lower()

The four guards to name: a **step/model-call limit** per run, a **tool-call limit** (overall and per expensive tool),
a **token or money budget** per run and per user, and a **wall-clock timeout** — plus alerting when a guard fires.

### A guardrail: `PIIMiddleware`

Redact personal data **before** it reaches the model provider (and your logs). A spy middleware shows what the model
actually received.

In [ ]:
from langchain.agents.middleware import PIIMiddleware, wrap_model_call

model_saw = []

@wrap_model_call
def spy(request, handler):
    model_saw.append(request.messages[0].content)
    return handler(request)

private = create_agent(llm, tools=[get_order_status],
                       middleware=[PIIMiddleware("email", strategy="redact", apply_to_input=True), spy])
private.invoke({"messages": [{"role": "user", "content": "My email is jane.doe@example.com - where is my order A5521?"}]})
print("model received:", model_saw[0])
assert "jane.doe@example.com" not in model_saw[0]

### Human in the loop: approve before an irreversible action

`HumanInTheLoopMiddleware(interrupt_on={"tool_name": True})` pauses the run **before** that tool executes. It needs a
checkpointer (the paused state has to be saved). The result contains `__interrupt__` with the proposed call; you
resume with `Command(resume={"decisions": [...]})` — `approve`, `edit` (change the arguments), `reject` or `respond`.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

sent = []

@tool
def send_email(to_email: str, body: str) -> str:
    """Send an email to a customer."""
    sent.append(to_email)
    return f"email sent to {to_email}"

guarded = create_agent(llm, tools=[send_email], checkpointer=InMemorySaver(),
                       middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True})])
cfg = {"configurable": {"thread_id": "refund-42"}}

paused = guarded.invoke({"messages": [{"role": "user", "content": "Send an email to bob@example.com saying the refund is approved"}]}, cfg)
request = paused["__interrupt__"][0].value["action_requests"][0]
print("paused before:", request["name"], request["args"]["to_email"], "| emails sent so far:", len(sent))
assert sent == []

done = guarded.invoke(Command(resume={"decisions": [{"type": "approve"}]}), cfg)
print("after approve:", done["messages"][-1].content, "| emails sent:", sent)
assert sent == ["bob@example.com"]

### Summarisation: bounded context — and the trap

`SummarizationMiddleware(model, trigger=("messages", 6), keep=("messages", 2))` replaces older messages with a
summary once the history reaches the trigger. The chart counts the messages the model receives per turn with and
without it. Then the trap (`pr11`, `ag15`): a summary is **lossy**, and a model decides what to drop. The offline
stand-in's "summary" is crude (the first sentences of the prompt it is given), so here it loses both the name and the
city — an exaggeration; a real model keeps more. But the lesson is real: pin facts that must never be lost (user id,
constraints) outside the summarised history, e.g. in the system prompt or a long-term store.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

def run_chat(middleware):
    seen = []

    @wrap_model_call
    def count(request, handler):
        seen.append(len(request.messages))
        return handler(request)

    bot = create_agent(llm, tools=[], middleware=[*middleware, count], checkpointer=InMemorySaver())
    cfg = {"configurable": {"thread_id": "long-chat"}}
    turns = ["My name is Sujan.", "My city is Pune.", "I like tennis.", "My team is payments.",
             "I use Python.", "I work remotely.", "What is my name?", "What is my city?"]
    replies = [bot.invoke({"messages": [{"role": "user", "content": t}]}, cfg)["messages"][-1].content for t in turns]
    return seen, replies

plain_seen, plain_replies = run_chat([])
summ_seen, summ_replies = run_chat([SummarizationMiddleware(model=llm, trigger=("messages", 6), keep=("messages", 2))])
print("messages sent per turn, no summary  :", plain_seen)
print("messages sent per turn, with summary:", summ_seen)
print("'What is my name?' ->", plain_replies[-2], "|", summ_replies[-2])
print("'What is my city?' ->", plain_replies[-1], "|", summ_replies[-1])
assert max(summ_seen) < max(plain_seen) and "Pune" in plain_replies[-1]

plt.figure(figsize=(8, 3.3))
plt.plot(range(1, 9), plain_seen, marker="o", label="full history")
plt.plot(range(1, 9), summ_seen, marker="o", label="SummarizationMiddleware")
plt.title("Messages sent to the model each turn"); plt.xlabel("turn"); plt.ylabel("messages in the request")
plt.legend(); plt.tight_layout(); plt.show()

## 5. Chat history for plain chains (legacy): `RunnableWithMessageHistory`

For a simple chain (no tools) the 0.x pattern wraps it with a per-session history store. It still works in 1.x but is
**deprecated** — both classes emit `LangChainDeprecationWarning` and point to LangGraph persistence (the checkpointer
above), and `InMemoryChatMessageHistory` is scheduled for removal in 2.0. You will meet it in existing code, so know it.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."), MessagesPlaceholder("history"), ("human", "{input}")])
chain = prompt | llm | StrOutputParser()

sessions = {}
def get_history(session_id: str) -> InMemoryChatMessageHistory:
    return sessions.setdefault(session_id, InMemoryChatMessageHistory())

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    chat = RunnableWithMessageHistory(chain, get_history, input_messages_key="input", history_messages_key="history")
    chat.invoke({"input": "My name is Alice."}, config={"configurable": {"session_id": "s1"}})
    print("s1:", chat.invoke({"input": "What is my name?"}, config={"configurable": {"session_id": "s1"}}))
    print("s2:", chat.invoke({"input": "What is my name?"}, config={"configurable": {"session_id": "s2"}}))

print("stored messages per session:", {k: len(v.messages) for k, v in sessions.items()})
print("deprecation warnings:", sorted({str(w.message)[:70] for w in caught if "deprecat" in str(w.message).lower()}))
assert any("deprecat" in str(w.message).lower() for w in caught)

Whatever stores the history, keep what you *send* bounded. `trim_messages` keeps the most recent messages that fit a
token budget, always keeping the system message and starting on a human turn (so no orphaned tool results):

In [ ]:
from langchain_core.messages import SystemMessage, trim_messages

long_history = [SystemMessage("You are helpful.")] + [
    m for i in range(6) for m in (HumanMessage(f"question {i}"), AIMessage(f"answer {i}"))]
kept = trim_messages(long_history, max_tokens=5, token_counter=len,       # len = count messages; use a tokenizer in production
                     strategy="last", include_system=True, start_on="human")
print([m.content for m in kept])
assert kept[0].type == "system" and kept[1].type == "human" and len(kept) == 5

## 6. When to drop down to LangGraph

`create_agent` *is* a LangGraph graph (section 1) with a fixed shape: model ⇄ tools. Stay with it while that shape
fits; write the graph yourself ([ReAct agent in LangGraph](../14_langgraph/05_react_tool_calling_agent.ipynb)) when:

| You need | Why `create_agent` is not enough |
|---|---|
| fixed steps around the loop (classify → retrieve → answer → validate) | the prebuilt graph only has model and tools nodes |
| branches, parallel workers, loops with custom exit rules | conditional edges between your own nodes |
| several agents (supervisor, handoffs) | [multi-agent supervisor](../14_langgraph/06_multi_agent_supervisor.ipynb) |
| custom state beyond messages that nodes read and write | typed state channels and reducers |
| interrupts at arbitrary points, not only before a tool | `interrupt()` inside any node ([human in the loop](../14_langgraph/04_human_in_the_loop_interrupts.ipynb)) |

Rule of thumb: a chain when the steps are known; `create_agent` when the model should choose tools in a loop;
LangGraph when *you* need to control the flow; and a hand-written loop (notebook 03, 15 lines) when you want zero
dependencies and full visibility.

## Try it yourself

**1. Cap one expensive tool.** Use `ToolCallLimitMiddleware(tool_name="search_policy", run_limit=0)` so the policy
tool can never run, and show what the agent answers instead.

In [ ]:
# Solution — try it yourself first, then run this
from langchain.agents.middleware import ToolCallLimitMiddleware

no_policy = create_agent(llm, tools=[get_order_status, search_policy],
                         middleware=[ToolCallLimitMiddleware(tool_name="search_policy", run_limit=0)])
run = no_policy.invoke({"messages": [{"role": "user", "content": "What is your refund policy?"}]})
show_trace(run["messages"])
tool_msgs = [m for m in run["messages"] if isinstance(m, ToolMessage)]
assert tool_msgs and "limit" in tool_msgs[0].content.lower()

**2. Reject the email.** Re-run the human-in-the-loop agent on a new thread and resume with a `reject` decision and a
message. Check that no email was sent.

In [ ]:
# Solution — try it yourself first, then run this
sent.clear()
cfg2 = {"configurable": {"thread_id": "refund-43"}}
guarded.invoke({"messages": [{"role": "user", "content": "Send an email to eve@example.com saying the refund is approved"}]}, cfg2)
after = guarded.invoke(Command(resume={"decisions": [{"type": "reject", "message": "Refund not approved yet."}]}), cfg2)
show_trace(after["messages"])
assert sent == []

**3. A per-user memory check.** Using `mem_agent`, tell thread `"carol-1"` "My city is Lisbon." and check that
`"carol-1"` knows the city while `"alice-1"` does not.

In [ ]:
# Solution — try it yourself first, then run this
carol = {"configurable": {"thread_id": "carol-1"}}
mem_agent.invoke({"messages": [{"role": "user", "content": "My city is Lisbon."}]}, carol)
c = mem_agent.invoke({"messages": [{"role": "user", "content": "What is my city?"}]}, carol)["messages"][-1].content
a = mem_agent.invoke({"messages": [{"role": "user", "content": "What is my city?"}]}, alice)["messages"][-1].content
print("carol-1:", c, "| alice-1:", a)
assert "Lisbon" in c and "Lisbon" not in a

## Say it in an interview

- **Agent vs chain:** a chain's steps are fixed; an agent's model picks the next action in a loop (model → tool calls
  → results → model) until it answers. `create_agent(model, tools, system_prompt)` is that loop as a LangGraph graph;
  `AgentExecutor` is the 0.x version (now in `langchain_classic`).
- **Read the trace:** Human → AI(tool_calls) → Tool → AI(answer). Every model call costs tokens; parallel tool calls
  save round trips. Stream with `stream_mode="updates"` for steps, `"messages"` for tokens.
- **Memory:** short-term = checkpointer + `thread_id` (InMemory for demos, Postgres/SQLite in production); long-term =
  a store; the context window = what survives trimming/summarising. "Forgot the beginning after 50 turns" = the
  history was trimmed or summarised (or overflowed) — pin must-keep facts outside it.
- **Middleware = the production layer:** call/tool limits (runaway cost), PII redaction, human approval before
  irreversible tools (with a checkpointer, resume via `Command`), summarisation, retries/fallbacks, custom hooks.
- **Guards against a $400 run:** model-call limit, tool-call limit, token/money budget, timeout — and alerts.
- **Legacy:** `RunnableWithMessageHistory` + `InMemoryChatMessageHistory` still work but are deprecated in favour of
  LangGraph persistence.
- **When LangGraph:** custom flow, custom state, multi-agent, interrupts anywhere. **When your own loop:** one simple
  agent where full visibility matters more than features.

## Next

- LangGraph, one level down: [state, nodes and edges](../14_langgraph/01_state_nodes_and_edges.ipynb) ·
  [ReAct tool-calling agent](../14_langgraph/05_react_tool_calling_agent.ipynb) ·
  [human in the loop](../14_langgraph/04_human_in_the_loop_interrupts.ipynb)
- Without a framework: [agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) ·
  [agent memory, short and long term](../15_agentic_ai/04_agent_memory_short_and_long_term.ipynb) ·
  [guardrails, HITL and failure modes](../15_agentic_ai/06_guardrails_hitl_and_failure_modes.ipynb) ·
  [build an agent in 30 minutes](../17_interview_coding_rounds/02_build_an_agent_in_30_minutes.ipynb)
- Theory: [LangChain course](../../langchain/index.html) (chapters 9–11) · [langgraph course](../../langgraph/index.html) ·
  [agentic_ai_flow](../../agentic_ai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)